# Medical Infectious-Disease RAG QA System

**Authors:** Christopher Protheroe and Tyler Burzenski  
**Project:** DSCI 6004 Natural Language Processing final project

This notebook is a reproducible companion for the project repository in `Medical-RAG-QA-System-main.zip`. It uses the repository code and included data to:

1. inspect the PubMed infectious-disease corpus,
2. validate preprocessing and chunking,
3. rebuild the ChromaDB vector index from the included processed chunks,
4. retrieve PubMed context for evaluation questions,
5. build the same grounded prompts used by the project,
6. optionally run the three Ollama-hosted open-source LLMs, and
7. analyze the included answer and RAGAS evaluation files.

> This is an NLP/RAG evaluation notebook, not a clinical decision-support tool. Generated answers should not be treated as medical advice.


## Run configuration

The notebook is designed to work in any of these layouts:

- opened from inside the extracted repository folder,
- opened from the same folder as `Medical-RAG-QA-System-main.zip`

Heavy cells are controlled by flags. By default, the notebook reads the included project results and does **not** call local LLMs. To regenerate model outputs, install Ollama, pull the three models listed below, start `ollama serve`, and set `RUN_FULL_EXPERIMENT = True` in the experiment section.


In [ ]:
from pathlib import Path
import json
import os
import sys
import zipfile
from collections import Counter

PROJECT_DIR_NAME = "Medical-RAG-QA-System-main"
ZIP_NAME = "Medical-RAG-QA-System-main.zip"


def looks_like_project(path: Path) -> bool:
    return (
        path.exists()
        and (path / "src" / "prompt.py").exists()
        and (path / "data" / "processed_chunks.json").exists()
        and (path / "requirements.txt").exists()
    )

candidate_roots = [
    Path.cwd(),
    Path.cwd() / PROJECT_DIR_NAME,
    Path("/mnt/data") / PROJECT_DIR_NAME,
    Path("/content") / PROJECT_DIR_NAME,
]

PROJECT_ROOT = next((p.resolve() for p in candidate_roots if looks_like_project(p)), None)

if PROJECT_ROOT is None:
    zip_candidates = [Path.cwd() / ZIP_NAME, Path("/mnt/data") / ZIP_NAME, Path("/content") / ZIP_NAME]
    zip_path = next((p for p in zip_candidates if p.exists()), None)
    if zip_path is None:
        raise FileNotFoundError(
            f"Could not find an extracted {PROJECT_DIR_NAME} folder or {ZIP_NAME}. "
            "Place the notebook next to the project zip or run it from the extracted repository."
        )
    extract_to = zip_path.parent
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    extracted = extract_to / PROJECT_DIR_NAME
    if not looks_like_project(extracted):
        raise FileNotFoundError(f"Extracted {zip_path}, but project structure was not found.")
    PROJECT_ROOT = extracted.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RAW_PATH = DATA_DIR / "raw_pubmed.json"
CHUNKS_PATH = DATA_DIR / "processed_chunks.json"
CHROMA_DIR = DATA_DIR / "chroma_db"
ANSWERS_PATH = RESULTS_DIR / "answers.csv"
RAGAS_PATH = RESULTS_DIR / "ragas_scores.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Source modules: {SRC_ROOT}")
print(f"Data directory: {DATA_DIR}")


## Dependencies

Core result-analysis cells need `pandas`. Retrieval/indexing cells need `chromadb` and `sentence-transformers`. Full model-generation cells need Ollama running locally plus the three model tags used by the project:

```bash
ollama pull llama3.1:8b-instruct-q4_K_M
ollama pull mistral:7b-instruct-q4_1
ollama pull phi3:mini
```

The install cell is disabled by default so the notebook does not unexpectedly change your environment. Enable it when running in a fresh environment.


In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])

# Lightweight dependency check. Missing packages are reported here; install from requirements.txt if needed.
import importlib.util

required_imports = {
    "pandas": "pandas",
    "chromadb": "chromadb",
    "sentence_transformers": "sentence-transformers",
    "requests": "requests",
    "Bio": "biopython",
}
missing = [pkg for import_name, pkg in required_imports.items() if importlib.util.find_spec(import_name) is None]

if missing:
    print("Missing packages:", ", ".join(missing))
    print("Set INSTALL_DEPENDENCIES = True above and rerun the cell, or install requirements.txt manually.")
else:
    print("Required core packages are importable.")


## Load the included corpus and result files

In [ ]:
import pandas as pd
from IPython.display import display

with RAW_PATH.open("r", encoding="utf-8") as f:
    raw_payload = json.load(f)

with CHUNKS_PATH.open("r", encoding="utf-8") as f:
    chunk_payload = json.load(f)

raw_records = raw_payload.get("records", [])
chunks = chunk_payload.get("chunks", [])

print(f"Raw PubMed records: {len(raw_records)}")
print(f"Processed chunks: {len(chunks)}")
print(f"Chunk size: {chunk_payload.get('chunk_size')} sentences; overlap: {chunk_payload.get('overlap')} sentence")

record_counts = pd.Series(Counter(r.get("topic", "") for r in raw_records), name="raw_records")
chunk_counts = pd.Series(Counter(c.get("metadata", {}).get("topic", "") for c in chunks), name="processed_chunks")
corpus_summary = pd.concat([record_counts, chunk_counts], axis=1).fillna(0).astype(int)
display(corpus_summary)

sample_chunk = chunks[0]
print("Sample chunk id:", sample_chunk["id"])
print("Sample title:", sample_chunk["metadata"].get("title"))
print("Sample text preview:")
print(sample_chunk["text"][:900])


## Validate preprocessing against the project source code

This cell imports the repository's `preprocess.py` functions and checks that the included `processed_chunks.json` is consistent with the first raw PubMed record.


In [ ]:
from preprocess import record_to_chunks

first_record = raw_records[0]
recomputed_first_record_chunks = record_to_chunks(
    first_record,
    chunk_size=chunk_payload.get("chunk_size", 4),
    overlap=chunk_payload.get("overlap", 1),
)

print(f"First PMID: {first_record.get('pmid')}")
print(f"Chunks recomputed from first record: {len(recomputed_first_record_chunks)}")
print(f"First recomputed chunk id: {recomputed_first_record_chunks[0]['id']}")
print(f"First stored chunk id:     {chunks[0]['id']}")

assert recomputed_first_record_chunks[0]["id"] == chunks[0]["id"], "Stored chunk id does not match preprocessing output."
assert recomputed_first_record_chunks[0]["text"] == chunks[0]["text"], "Stored chunk text does not match preprocessing output."
print("Preprocessing validation passed.")


## Project questions and model configuration

These are the 10 domain-specific questions and three model tags used by the repository's `run_models.py` experiment runner.


In [ ]:
DEFAULT_MODELS = [
    "llama3.1:8b-instruct-q4_K_M",
    "mistral:7b-instruct-q4_1",
    "phi3:mini",
]

DEFAULT_QUESTIONS = [
    "What evidence in the corpus describes common symptoms of COVID-19?",
    "What does the corpus say about influenza vaccine effectiveness?",
    "How is latent tuberculosis different from active tuberculosis?",
    "What treatments are discussed for HIV infection?",
    "What complications are associated with severe COVID-19 cases?",
    "What diagnostic approaches are mentioned for tuberculosis?",
    "What risk factors are associated with influenza severity?",
    "What does the corpus say about antiviral therapy for HIV?",
    "How do the retrieved studies describe prevention strategies for tuberculosis spread?",
    "What limitations or uncertainties are reported in the corpus for infectious disease treatment?",
]

questions_df = pd.DataFrame({"question_id": range(1, len(DEFAULT_QUESTIONS) + 1), "question": DEFAULT_QUESTIONS})
display(questions_df)
print("Models:")
for model in DEFAULT_MODELS:
    print("-", model)


## Build or load the ChromaDB vector index

The project zip includes `processed_chunks.json` but not a persisted `data/chroma_db/` directory. This cell rebuilds the index with the same embedding model named in the project code: `sentence-transformers/all-MiniLM-L6-v2`.

The helper below intentionally serializes list-valued metadata fields such as `authors` and `mesh_terms` before insertion. This makes the notebook robust across ChromaDB versions, since Chroma metadata values are expected to be scalar values.


In [ ]:
DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "pubmed_medical_rag"


def _collection_names(client):
    names = []
    for item in client.list_collections():
        names.append(item.name if hasattr(item, "name") else str(item))
    return set(names)


def chroma_safe_metadata(metadata: dict) -> dict:
    safe = {}
    for key, value in metadata.items():
        if value is None or isinstance(value, (str, int, float, bool)):
            safe[key] = value
        elif isinstance(value, (list, tuple, dict)):
            safe[key] = json.dumps(value, ensure_ascii=False)
        else:
            safe[key] = str(value)
    return safe


def build_or_load_chroma_index(
    chunks: list[dict],
    persist_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
    embedding_model: str = DEFAULT_EMBEDDING_MODEL,
    reset: bool = False,
):
    import shutil
    import chromadb
    from chromadb.utils import embedding_functions

    if reset and persist_dir.exists():
        shutil.rmtree(persist_dir)
    persist_dir.mkdir(parents=True, exist_ok=True)

    embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embedding_model)
    client = chromadb.PersistentClient(path=str(persist_dir))
    existing = _collection_names(client)

    if collection_name in existing:
        collection = client.get_collection(name=collection_name, embedding_function=embedding_fn)
        if collection.count() == len(chunks):
            print(f"Loaded existing Chroma collection '{collection_name}' with {collection.count()} chunks.")
            return collection
        client.delete_collection(collection_name)

    collection = client.create_collection(name=collection_name, embedding_function=embedding_fn)
    ids = [item["id"] for item in chunks]
    docs = [item["text"] for item in chunks]
    metadatas = [chroma_safe_metadata(item.get("metadata", {})) for item in chunks]

    batch_size = 128
    for start in range(0, len(chunks), batch_size):
        end = start + batch_size
        collection.add(
            ids=ids[start:end],
            documents=docs[start:end],
            metadatas=metadatas[start:end],
        )
    print(f"Indexed {collection.count()} chunks into '{collection_name}' at {persist_dir}.")
    return collection


RESET_INDEX = False
collection = build_or_load_chroma_index(chunks, reset=RESET_INDEX)


## Retrieval and grounded prompt construction

In [ ]:
from prompt import build_prompt, format_context, SYSTEM_PROMPT

TOPIC_HINTS = {
    "covid": "covid19",
    "sars-cov-2": "covid19",
    "influenza": "influenza",
    "flu": "influenza",
    "tuberculosis": "tuberculosis",
    "tb": "tuberculosis",
    "hiv": "hiv",
}


def infer_topic_filter(question: str) -> str | None:
    q = question.lower()
    for needle, topic in TOPIC_HINTS.items():
        if needle in q:
            return topic
    return None


def retrieve_chunks_notebook(
    query: str,
    top_k: int = 3,
    topic_filter: str | None = None,
    collection=collection,
) -> list[dict]:
    where = {"topic": topic_filter} if topic_filter else None
    results = collection.query(query_texts=[query], n_results=top_k, where=where)
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]
    ids = results.get("ids", [[]])[0]
    return [
        {"id": doc_id, "document": doc, "metadata": meta, "distance": dist}
        for doc_id, doc, meta, dist in zip(ids, docs, metas, dists)
    ]

SAMPLE_QUESTION = DEFAULT_QUESTIONS[1]
sample_topic = infer_topic_filter(SAMPLE_QUESTION)
sample_retrieved = retrieve_chunks_notebook(SAMPLE_QUESTION, top_k=3, topic_filter=sample_topic)

retrieval_preview = pd.DataFrame([
    {
        "rank": i,
        "id": r["id"],
        "distance": r["distance"],
        "topic": r["metadata"].get("topic"),
        "year": r["metadata"].get("year"),
        "title": r["metadata"].get("title"),
    }
    for i, r in enumerate(sample_retrieved, start=1)
])
display(retrieval_preview)

sample_prompt = build_prompt(SAMPLE_QUESTION, sample_retrieved)
print(sample_prompt[:3500])


## Optional: query one local Ollama model

This cell uses the same prompt and Ollama HTTP call style as `src/run_models.py`. It is disabled by default because it requires a local Ollama server and the model weights.


In [ ]:
import requests
import time

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")


def ollama_available(base_url: str = OLLAMA_BASE_URL) -> bool:
    try:
        response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
        response.raise_for_status()
        return True
    except Exception:
        return False


def query_ollama(model: str, prompt: str, base_url: str = OLLAMA_BASE_URL, timeout: int = 300) -> str:
    url = f"{base_url.rstrip('/')}/api/generate"
    response = requests.post(
        url,
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json().get("response", "").strip()

RUN_SINGLE_MODEL = False
SINGLE_MODEL = "phi3:mini"

if RUN_SINGLE_MODEL:
    if not ollama_available():
        raise RuntimeError(f"Ollama is not reachable at {OLLAMA_BASE_URL}. Start `ollama serve` first.")
    started = time.time()
    answer = query_ollama(SINGLE_MODEL, sample_prompt)
    print(f"Model: {SINGLE_MODEL}")
    print(f"Latency: {time.time() - started:.2f} seconds")
    print(answer)
else:
    print("Set RUN_SINGLE_MODEL = True to generate a live answer with Ollama.")


## Optional: regenerate the full three-model experiment

The included `results/answers.csv` already contains 30 rows: 10 questions × 3 models. Leave `RUN_FULL_EXPERIMENT = False` to analyze the included run. Set it to `True` only when Ollama is running and all three models are pulled locally.


In [ ]:
def run_experiment_notebook(
    questions: list[str],
    models: list[str],
    top_k: int = 3,
    output_path: Path = ANSWERS_PATH,
) -> pd.DataFrame:
    rows = []
    for question in questions:
        topic_filter = infer_topic_filter(question)
        retrieved = retrieve_chunks_notebook(question, top_k=top_k, topic_filter=topic_filter)
        prompt = build_prompt(question, retrieved)
        for model in models:
            started = time.time()
            try:
                answer = query_ollama(model, prompt)
                error = ""
            except Exception as exc:
                answer = ""
                error = str(exc)
            rows.append(
                {
                    "question": question,
                    "topic_filter": topic_filter or "",
                    "model": model,
                    "latency_seconds": round(time.time() - started, 3),
                    "answer": answer,
                    "error": error,
                    "retrieved_context": json.dumps(retrieved, ensure_ascii=False),
                }
            )
    df = pd.DataFrame(rows)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    return df

RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    if not ollama_available():
        raise RuntimeError(f"Ollama is not reachable at {OLLAMA_BASE_URL}. Start `ollama serve` first.")
    answers_df = run_experiment_notebook(DEFAULT_QUESTIONS, DEFAULT_MODELS)
    print(f"Generated and saved {len(answers_df)} rows to {ANSWERS_PATH}")
else:
    answers_df = pd.read_csv(ANSWERS_PATH)
    print(f"Loaded included answers: {ANSWERS_PATH}")
    print(f"Rows: {len(answers_df)}")

display(answers_df[["question", "topic_filter", "model", "latency_seconds", "error"]].head(10))


## Analyze included answer and RAGAS score files

In [ ]:
ragas_df = pd.read_csv(RAGAS_PATH)
print(f"Loaded RAGAS scores: {RAGAS_PATH}")
print(f"RAGAS rows: {len(ragas_df)}")

# The project RAGAS CSV does not include model names, but it is row-aligned with answers.csv.
if "model" not in ragas_df.columns and len(ragas_df) == len(answers_df):
    scored_df = pd.concat(
        [answers_df[["question", "topic_filter", "model", "latency_seconds", "error"]].reset_index(drop=True), ragas_df.reset_index(drop=True)],
        axis=1,
    )
else:
    scored_df = ragas_df.copy()

for metric in ["faithfulness", "answer_relevancy"]:
    if metric in scored_df.columns:
        scored_df[metric] = pd.to_numeric(scored_df[metric], errors="coerce")

summary_by_model = (
    scored_df.groupby("model", dropna=False)
    .agg(
        responses=("model", "size"),
        mean_latency_seconds=("latency_seconds", "mean"),
        mean_faithfulness=("faithfulness", "mean"),
        min_faithfulness=("faithfulness", "min"),
        mean_answer_relevancy=("answer_relevancy", "mean"),
        min_answer_relevancy=("answer_relevancy", "min"),
        errors=("error", lambda s: int((s.fillna("") != "").sum())),
    )
    .sort_values(["mean_faithfulness", "mean_answer_relevancy"], ascending=False)
)

display(summary_by_model.style.format({
    "mean_latency_seconds": "{:.2f}",
    "mean_faithfulness": "{:.3f}",
    "min_faithfulness": "{:.3f}",
    "mean_answer_relevancy": "{:.3f}",
    "min_answer_relevancy": "{:.3f}",
}))

overall = scored_df[["faithfulness", "answer_relevancy"]].agg(["count", "mean", "min", "max"]).T
display(overall.style.format("{:.3f}"))


## Inspect strongest and weakest scored responses

In [ ]:
review_columns = [
    "model",
    "question",
    "faithfulness",
    "answer_relevancy",
    "response" if "response" in scored_df.columns else "answer",
]
review_columns = [c for c in review_columns if c in scored_df.columns]

print("Lowest faithfulness rows:")
display(scored_df.sort_values("faithfulness", ascending=True)[review_columns].head(5))

print("Lowest answer relevancy rows:")
display(scored_df.sort_values("answer_relevancy", ascending=True)[review_columns].head(5))

print("Highest combined score rows:")
scored_df["combined_score"] = scored_df[["faithfulness", "answer_relevancy"]].mean(axis=1)
display(scored_df.sort_values("combined_score", ascending=False)[review_columns + ["combined_score"]].head(5))


## Retrieval context audit

This section parses the stored retrieval contexts from `answers.csv` and checks which topics were retrieved for each question/model pair.


In [ ]:
def parse_retrieved_context(raw: str) -> list[dict]:
    try:
        parsed = json.loads(raw)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

context_rows = []
for row in answers_df.to_dict(orient="records"):
    retrieved = parse_retrieved_context(row.get("retrieved_context", ""))
    context_rows.append(
        {
            "question": row.get("question"),
            "model": row.get("model"),
            "topic_filter": row.get("topic_filter"),
            "num_retrieved_chunks": len(retrieved),
            "retrieved_topics": ", ".join(sorted({str(c.get("metadata", {}).get("topic", "")) for c in retrieved})),
            "retrieved_pmids": ", ".join([str(c.get("metadata", {}).get("pmid", "")) for c in retrieved]),
        }
    )

context_audit_df = pd.DataFrame(context_rows)
display(context_audit_df.head(15))

context_summary = context_audit_df.groupby(["topic_filter", "retrieved_topics"]).size().reset_index(name="rows")
display(context_summary)


## Optional: rerun RAGAS evaluation

The repository's `src/evaluate.py` uses local Ollama for the evaluator LLM and `nomic-embed-text` embeddings. This cell is disabled by default because it requires Ollama plus the evaluator model and embedding model.


In [ ]:
RUN_RAGAS_EVALUATION = False

if RUN_RAGAS_EVALUATION:
    from evaluate import maybe_run_ragas

    ok = maybe_run_ragas(str(ANSWERS_PATH), str(RAGAS_PATH))
    if ok:
        print(f"RAGAS results written to {RAGAS_PATH}")
    else:
        print("RAGAS evaluation did not run. Check dependencies and Ollama model availability.")
else:
    print("Set RUN_RAGAS_EVALUATION = True to rerun the local RAGAS evaluation.")


## Optional: rebuild corpus from PubMed

The included repository already contains `data/raw_pubmed.json` and `data/processed_chunks.json`. Use this section only if you want to refetch PubMed records and regenerate chunks.

NCBI Entrez requires an email address. An API key is optional.


In [ ]:
RUN_PUBMED_FETCH = False
NCBI_EMAIL = os.getenv("NCBI_EMAIL", "")
NCBI_API_KEY = os.getenv("NCBI_API_KEY", "")

if RUN_PUBMED_FETCH:
    if not NCBI_EMAIL:
        raise ValueError("Set NCBI_EMAIL in your environment before fetching PubMed records.")
    import subprocess

    fetch_cmd = [
        sys.executable,
        str(SRC_ROOT / "fetch_pubmed.py"),
        "--email",
        NCBI_EMAIL,
        "--per-topic",
        "25",
        "--output",
        str(RAW_PATH),
    ]
    if NCBI_API_KEY:
        fetch_cmd.extend(["--api-key", NCBI_API_KEY])

    subprocess.check_call(fetch_cmd, cwd=str(PROJECT_ROOT))
    subprocess.check_call(
        [
            sys.executable,
            str(SRC_ROOT / "preprocess.py"),
            "--input",
            str(RAW_PATH),
            "--output",
            str(CHUNKS_PATH),
            "--chunk-size",
            "4",
            "--overlap",
            "1",
        ],
        cwd=str(PROJECT_ROOT),
    )
    print("PubMed corpus and processed chunks regenerated. Re-run the index-building cell next.")
else:
    print("Set RUN_PUBMED_FETCH = True to refetch the PubMed corpus.")


##Notebook Summary

This notebook presents the full Medical RAG Question Answering System project. The project implements a Retrieval-Augmented Generation pipeline for answering infectious disease questions using a PubMed-based medical corpus. It focuses on evaluating how well open-source language models can answer domain-specific medical questions when grounded in retrieved biomedical evidence.

The notebook begins by setting up the project environment and loading the infectious disease corpus. The corpus consists of PubMed abstracts related to topics such as COVID-19, influenza, tuberculosis, HIV, and other infectious diseases. These documents are cleaned, processed, and split into smaller text chunks so they can be embedded and searched efficiently.

The core of the notebook builds the RAG pipeline. It uses sentence-transformer embeddings to convert the processed PubMed chunks into vector representations, then stores them in a ChromaDB vector database. When a user asks a medical question, the notebook retrieves the most relevant chunks from the vector database and passes them into a structured prompt. This prompt instructs the language model to answer using only the provided medical context, helping reduce hallucinations and improve factual grounding.

The notebook evaluates three open-source language models:

- Llama-3-8B
- Mistral-7B
- Phi-3-mini

Each model is tested on the same set of domain-specific infectious disease questions. For every question, the notebook records the retrieved context, generated answer, and evaluation results. This allows the models to be compared under the same retrieval conditions.

The evaluation section analyzes model performance using the project's generated answer file and RAGAS scoring results. The notebook compares the models across key dimensions such as answer faithfulness, context relevance, factual grounding, and overall response quality. It also includes inspection steps for reviewing retrieved passages and identifying where responses may be unsupported, incomplete, or hallucinated.

Overall, the notebook serves as the complete implementation and analysis environment for the Medical RAG QA System. It combines corpus preparation, vector indexing, retrieval, generation, evaluation, and comparative model analysis in a single executable workflow. Its purpose is to determine whether smaller free or open-source language models can produce reliable medical answers when supported by a retrieval-augmented PubMed knowledge base.
